# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_2_DEMO_Session_2_langgraph_memory_types.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





# LangGraph memory: one production-shaped agent

This notebook is built as **one runnable application**: a single graph, one **`BaseStore`**, and one **checkpointer**. The only “production deltas” should be **swapping persistence backends** (SQLite / Postgres) and **wiring your HTTP layer** to the same `UserMemoryService` helpers users would call from a settings screen.

You will see:

- **Short-term** memory: conversation state in `messages` + checkpointer / `thread_id`.
- **Long-term** memory: semantic facts, episodic examples, and procedural instructions in namespaced store keys.
- **Governance**: **list / edit / delete / clear** (what you expose in product UI), plus **summarize the thread** and **compact semantic memories** so nothing gets permanently crowded or messy.
- **Persistence, encryption, time travel**: same mechanisms as before, applied to this app.

Persisted state can still be abused via **memory poisoning**; use access control, validation, and optional **`EncryptedSerializer`** + custom **`CipherProtocol`** implementations ([persistence docs](https://docs.langchain.com/oss/python/langgraph/persistence)).

**References:** [Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory) · [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) · [Time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel) · [llms.txt](https://docs.langchain.com/llms.txt)


# Important Notes on Memory

Remember that users *must* be able to see and edit what the AI remembers about them. This is important for trust and GDPR/privacy compliance.

You usually have a *"thre-legged stool"* of agent state:

- __Short-term:__ What’s happening right now (Threads).
- __Long-term (Semantic):__ What I know about you (Store).
- __Procedural:__ How you told me to behave (Instructions).


---


### 1. The "Service Layer" Pattern

The use of `UserMemoryService` is important in production.

* It decouples the memory logic from the Graph nodes.
* You can call `svc.list_semantic()` from a FastAPI endpoint (for a user settings page) and from inside a Graph node using the same logic. This prevents "logic drift" where the agent sees memory differently than your user does.

### 2. Memory Hygiene (Summarization & Compaction)
Don't ignore what happens after 1,000 messages, you need to addresses it via:

* **Thread Summarization:** Using `RemoveMessage` to prune the chat history while keeping the "gist".
* **Semantic Compaction:** The `/memory compact` command takes 20 individual facts (e.g., "User likes coffee," "User likes lattes") and merges them into one consolidated profile. This saves tokens and reduces "distraction" for the agent.

### 3. Namespace Strategy
Use `(user_id, "semantic")` or `(user_id, "procedural")` as namespaces, this is the industry standard for multi-tenant apps. It ensures:

* **Privacy:** User A cannot access User B's memories.
* **Organization:** You can clear "episodic" memories (past examples) without losing "procedural" memories (how the user wants the agent to behave).

In [ ]:
# Production delta: bump versions as needed, then restart kernel if prompted.
%pip install -q "langgraph>=0.2" "langchain>=0.3" "langchain-openai>=0.3" "langchain-core>=0.3" "python-dotenv>=1.0" "pycryptodome>=3.20"


In [ ]:
from __future__ import annotations

import os
import re
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from operator import add
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
)
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from langgraph.runtime import Runtime
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")

llm: ChatOpenAI | None
if OPENAI_API_KEY:
    llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    print(f"LLM enabled ({OPENAI_MODEL}) — embeddings + summarization use OpenAI.")
else:
    llm = None
    print("No OPENAI_API_KEY: chat/summaries use small offline heuristics; store uses a deterministic embedding stub.")


# --- Production swap: checkpointer backend (one line change) -----------------
def build_checkpointer():
    """Return a BaseCheckpointSaver. Replace InMemorySaver for durability."""
    # from langgraph.checkpoint.sqlite import SqliteSaver
    # import sqlite3
    # conn = sqlite3.connect("langgraph_checkpoints.db", check_same_thread=False)
    # return SqliteSaver(conn)
    # from langgraph.checkpoint.postgres import PostgresSaver
    # cp = PostgresSaver.from_conn_string(os.environ["DATABASE_URL"]); cp.setup(); return cp
    return InMemorySaver()


# --- Production swap: store backend + real embeddings ----------------------
def _stub_embed(texts: list[str]) -> list[list[float]]:
    """Deterministic pseudo-vectors for CI / no-key runs."""
    out: list[list[float]] = []
    for t in texts:
        n = min(8, max(1, len(t) % 8))
        vec = [float((ord(c) % 13) / 13.0) for c in t[:32]]
        while len(vec) < 8:
            vec.append(0.0)
        out.append(vec[:8])
    return out


def build_store() -> InMemoryStore:
    """Long-term memory index. With OPENAI_API_KEY, uses text-embedding-3-small."""
    if OPENAI_API_KEY:
        from langchain.embeddings import init_embeddings

        embedder = init_embeddings("openai:text-embedding-3-small")
        return InMemoryStore(
            index={
                "embed": embedder,
                "dims": 1536,
                "fields": ["fact", "task", "outcome", "instructions", "text", "$"],
            }
        )
    return InMemoryStore(index={"embed": _stub_embed, "dims": 8})


NS_SEMANTIC = "semantic"
NS_EPISODIC = "episodic"
NS_PROCEDURAL = "procedural"
PROFILE_KEY = "agent_instructions"
CONSOLIDATED_KEY = "consolidated_profile"


@dataclass(frozen=True)
class AppContext:
    user_id: str


def ns(ctx: AppContext, kind: str) -> tuple[str, str]:
    return (ctx.user_id, kind)


def _iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def _ensure_msg_ids(messages: list[BaseMessage]) -> list[BaseMessage]:
    fixed: list[BaseMessage] = []
    for m in messages:
        mid = getattr(m, "id", None)
        if mid is None:
            fixed.append(m.model_copy(update={"id": str(uuid.uuid4())}))
        else:
            fixed.append(m)
    return fixed


class UserMemoryService:
    """App-layer API: expose these methods from FastAPI / Next.js so users can manage memory.

    The agent nodes call the same helpers — one implementation, no drift.
    """

    def __init__(self, store: BaseStore):
        self.store = store

    def list_semantic(self, ctx: AppContext, limit: int = 50):
        return list(self.store.search(ns(ctx, NS_SEMANTIC), limit=limit))

    def list_episodic(self, ctx: AppContext, limit: int = 50):
        return list(self.store.search(ns(ctx, NS_EPISODIC), limit=limit))

    def get_procedural(self, ctx: AppContext):
        return self.store.get(ns(ctx, NS_PROCEDURAL), PROFILE_KEY)

    def delete_memory(self, ctx: AppContext, kind: str, key: str) -> None:
        if kind not in {NS_SEMANTIC, NS_EPISODIC}:
            raise ValueError("delete only supported for semantic/episodic keys here")
        self.store.delete(ns(ctx, kind), key)

    def clear_namespace(self, ctx: AppContext, kind: str) -> int:
        """Remove all keys in a namespace (user clicked “forget my facts”). Returns delete count."""
        nspath = ns(ctx, kind)
        items = list(self.store.search(nspath, limit=500))
        for it in items:
            self.store.delete(tuple(it.namespace), it.key)
        return len(items)

    def upsert_semantic(self, ctx: AppContext, key: str, fact: str, extra: dict | None = None) -> None:
        payload = {"fact": fact, "updated_at": _iso(), **(extra or {})}
        self.store.put(ns(ctx, NS_SEMANTIC), key, payload)

    def set_procedural(self, ctx: AppContext, instructions: str) -> None:
        self.store.put(
            ns(ctx, NS_PROCEDURAL),
            PROFILE_KEY,
            {"instructions": instructions, "updated_at": _iso()},
        )


# Shared instances for this notebook (= one deployed app)
CHECKPOINTER = build_checkpointer()
STORE = build_store()
MEMORY = UserMemoryService(STORE)


def _default_procedural(runtime: Runtime[AppContext]) -> None:
    n = ns(runtime.context, NS_PROCEDURAL)
    if runtime.store.get(n, PROFILE_KEY) is None:
        runtime.store.put(
            n,
            PROFILE_KEY,
            {
                "instructions": "Be concise and helpful. Prefer bullet points for comparisons.",
                "updated_at": _iso(),
            },
        )


def _last_human(state: MessagesState) -> HumanMessage | None:
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            return m
    return None


def route_entry(state: MessagesState) -> Literal["commands", "agent"]:
    h = _last_human(state)
    if h and isinstance(h.content, str) and h.content.strip().startswith("/"):
        return "commands"
    return "agent"


def node_commands(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    """Slash commands map 1:1 to product actions (settings UI, support tools)."""
    h = _last_human(state)
    if not h or not isinstance(h.content, str):
        return {}
    line = h.content.strip()
    ctx = runtime.context
    svc = MEMORY

    # /memory list
    if line == "/memory list" or line.startswith("/memory list"):
        sem = svc.list_semantic(ctx)
        epi = svc.list_episodic(ctx)
        proc = runtime.store.get(ns(ctx, NS_PROCEDURAL), PROFILE_KEY)
        lines = ["**Your memories (stored for this user_id)**"]
        lines.append(f"- Semantic ({len(sem)}): " + ", ".join(f"{it.key[:8]}…" for it in sem[:10]) + (" …" if len(sem) > 10 else ""))
        for it in sem[:5]:
            lines.append(f"    • [{it.key}] {it.value.get('fact', it.value)}")
        lines.append(f"- Episodic ({len(epi)}): " + ", ".join(it.key for it in epi[:5]))
        for it in epi[:3]:
            lines.append(f"    • {it.value.get('task')} -> {it.value.get('outcome')}")
        if proc:
            lines.append("- Procedural: " + str(proc.value.get("instructions", ""))[:200])
        return {"messages": [AIMessage(content="\n".join(lines))]}

    # /memory delete semantic <key>  |  /memory delete episodic <key>
    m = re.match(r"/memory\s+delete\s+(semantic|episodic)\s+(\S+)\s*$", line, re.I)
    if m:
        kind, key = m.group(1).lower(), m.group(2)
        svc.delete_memory(ctx, kind, key)
        return {"messages": [AIMessage(content=f"Deleted {kind} memory `{key}`.")]}

    # /memory clear semantic|episodic|all
    m = re.match(r"/memory\s+clear\s+(semantic|episodic|all)\s*$", line, re.I)
    if m:
        scope = m.group(1).lower()
        n = 0
        if scope in ("semantic", "all"):
            n += svc.clear_namespace(ctx, NS_SEMANTIC)
        if scope in ("episodic", "all"):
            n += svc.clear_namespace(ctx, NS_EPISODIC)
        return {"messages": [AIMessage(content=f"Cleared store entries ({n} removed).")]}

    # /memory edit semantic <key> <new fact...>
    m = re.match(r"/memory\s+edit\s+semantic\s+(\S+)\s+(.+)$", line, re.I | re.S)
    if m:
        key, new_fact = m.group(1), m.group(2).strip()
        svc.upsert_semantic(ctx, key, new_fact)
        return {"messages": [AIMessage(content=f"Updated semantic memory `{key}`.")]}

    # /proc set <instructions...>
    m = re.match(r"/proc\s+set\s+(.+)$", line, re.I | re.S)
    if m:
        svc.set_procedural(ctx, m.group(1).strip())
        return {"messages": [AIMessage(content="Updated procedural / style instructions.")]}

    if line.strip() == "/proc reset":
        runtime.store.delete(ns(ctx, NS_PROCEDURAL), PROFILE_KEY)
        return {
            "messages": [
                AIMessage(content="Procedural instructions removed; defaults apply on the next normal message.")
            ]
        }

    # /thread summarize — compress short-term transcript
    if line.startswith("/thread summarize"):
        msgs = _ensure_msg_ids(list(state["messages"]))
        # Drop the command message from summarization payload
        body = [m for m in msgs if m is not h]
        if llm is None:
            summary = "Summary (offline): " + " | ".join(
                str(getattr(x, "content", ""))[:80] for x in body[-6:] if getattr(x, "content", None)
            )
        else:
            transcript = "\n".join(f"{m.type}: {m.content}" for m in body if isinstance(m.content, str))
            summary = llm.invoke(
                [
                    SystemMessage(
                        content="Summarize the conversation for future context in <=8 bullet points. "
                        "Preserve user preferences and decisions. No preamble."
                    ),
                    HumanMessage(content=transcript),
                ]
            ).content
        removes = [RemoveMessage(id=m.id) for m in msgs if m.id]
        new_human = HumanMessage(
            id=str(uuid.uuid4()),
            content=f"Prior conversation (summarized):\n{summary}",
        )
        return {"messages": removes + [new_human]}

    # /memory compact — merge many semantic atoms into one profile blob + optional cleanup
    if line.startswith("/memory compact"):
        items = svc.list_semantic(ctx, limit=200)
        if not items:
            return {"messages": [AIMessage(content="No semantic memories to compact.")]}
        if llm is None:
            merged = "; ".join(it.value.get("fact", str(it.value)) for it in items[:20])
        else:
            facts = "\n".join(f"- ({it.key}) {it.value.get('fact', it.value)}" for it in items)
            merged = llm.invoke(
                [
                    SystemMessage(
                        content="Merge these user-specific facts into one structured profile (markdown bullets). "
                        "De-duplicate aggressively. Drop stale contradictions keeping the newest updated_at if present."
                    ),
                    HumanMessage(content=facts),
                ]
            ).content
        runtime.store.put(
            ns(ctx, NS_SEMANTIC),
            CONSOLIDATED_KEY,
            {"fact": merged, "updated_at": _iso(), "source": "compact"},
        )
        deleted = 0
        for it in items:
            if it.key == CONSOLIDATED_KEY:
                continue
            runtime.store.delete(tuple(it.namespace), it.key)
            deleted += 1
        return {
            "messages": [
                AIMessage(
                    content=f"Compacted semantic memories into `{CONSOLIDATED_KEY}` and removed {deleted} prior keys."
                )
            ]
        }

    return {"messages": [AIMessage(content=f"Unknown command: {line!r}. Try /memory list")]}


def node_agent(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    """Build system context from the store each turn (not persisted as graph messages)."""
    _default_procedural(runtime)
    ctx = runtime.context
    h = _last_human(state)
    q = (h.content if h and isinstance(h.content, str) else "") or ""
    sem = list(runtime.store.search(ns(ctx, NS_SEMANTIC), query=q or "user preferences", limit=6))
    epi = list(runtime.store.search(ns(ctx, NS_EPISODIC), query=q or "similar task", limit=3))
    proc = runtime.store.get(ns(ctx, NS_PROCEDURAL), PROFILE_KEY)
    assert proc is not None
    facts = "\n".join(f"- {s.value.get('fact', s.value)}" for s in sem) or "- (none)"
    eps = "\n".join(
        f"- task: {e.value.get('task')} => outcome: {e.value.get('outcome')}" for e in epi
    ) or "- (none)"
    sys = SystemMessage(
        content=(
            "You are a single assistant with long-term memory loaded from a store.\n"
            f"Procedural instructions:\n{proc.value['instructions']}\n\n"
            f"Semantic facts:\n{facts}\n\n"
            f"Relevant episodic examples:\n{eps}\n\n"
            "Users can say `remember: ...` to store a fact, or `episode: <task> | <outcome>` to store an experience. "
            "They can manage memory with slash commands like `/memory list`."
        ),
    )
    conv = [m for m in state["messages"] if isinstance(m, (HumanMessage, AIMessage))]
    if llm is None:
        return {"messages": [AIMessage(content=f"Heuristic reply (no API key): you said {q!r}")]}
    ai = llm.invoke([sys, *conv])
    return {"messages": [ai]}


def node_persist(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    """Hot-path writes: same hooks a production agent would use after a successful turn."""
    msgs = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    if len(msgs) < 2:
        return {}
    human = None
    ai = None
    for m in reversed(msgs):
        if ai is None and isinstance(m, AIMessage):
            ai = m
        elif human is None and isinstance(m, HumanMessage):
            human = m
            break
    if human is None or ai is None or not isinstance(human.content, str):
        return {}
    text = human.content.strip()
    if text.startswith("/"):
        return {}
    ctx = runtime.context

    m = re.search(r"remember\s*:\s*(.+)", text, flags=re.I | re.S)
    if m:
        key = str(uuid.uuid4())
        MEMORY.upsert_semantic(ctx, key, m.group(1).strip())

    m = re.search(r"episode\s*:\s*(.+?)\s*\|\s*(.+)", text, flags=re.I | re.S)
    if m:
        runtime.store.put(
            ns(ctx, NS_EPISODIC),
            str(uuid.uuid4()),
            {
                "task": m.group(1).strip(),
                "outcome": m.group(2).strip(),
                "created_at": _iso(),
            },
        )

    return {}


workflow = StateGraph(MessagesState, context_schema=AppContext)
workflow.add_node("commands", node_commands)
workflow.add_node("agent", node_agent)
workflow.add_node("persist", node_persist)

workflow.add_conditional_edges(START, route_entry, {"commands": "commands", "agent": "agent"})
workflow.add_edge("commands", END)
workflow.add_edge("agent", "persist")
workflow.add_edge("persist", END)

agent_app = workflow.compile(checkpointer=CHECKPOINTER, store=STORE)
print("Compiled unified agent_app (checkpointer + store wired).")


LLM enabled (gpt-5.4-nano) — embeddings + summarization use OpenAI.
Compiled unified agent_app (checkpointer + store wired).


## Runbook: same app, realistic user + product flows

The **slash commands** are stand-ins for **authenticated REST routes** (`GET /memory`, `DELETE /memory/{kind}/{key}`, etc.). In production, pass the same `user_id` you use in `AppContext` after verifying the session.

**Short-term hygiene:** `/thread summarize` collapses noisy chat history while keeping a checkpointed thread (uses `RemoveMessage` under the hood).

**Long-term hygiene:** `/memory compact` merges many semantic entries into `consolidated_profile` and deletes the old keys (tune retention in production: archive instead of hard-delete if you need audit).

**User trust:** expose **list / edit / delete / clear** so people can correct mistakes — that also reduces poisoned or stale memories driving the model.

| Command | Product equivalent |
| --- | --- |
| `/memory list` | Memory settings screen |
| `/memory edit semantic <key> <text>` | Inline edit of one fact |
| `/memory delete semantic|episodic <key>` | Per-item delete |
| `/memory clear semantic|episodic|all` | “Forget” category or everything |
| `/proc set ...` | “How should the assistant behave?” |
| `/proc reset` | Drop custom instructions (re-seed defaults) |
| `/thread summarize` | Compress noisy chat (short-term) |
| `/memory compact` | Merge many facts into one profile (long-term) |


In [ ]:
def run_turn(thread: str, user: str, text: str):
    cfg: RunnableConfig = {"configurable": {"thread_id": thread}}
    return agent_app.invoke(
        {"messages": [HumanMessage(content=text, id=str(uuid.uuid4()))]},
        cfg,
        context=AppContext(user_id=user),
    )


USER = "demo-user-1"
THREAD = "demo-thread-1"

run_turn(THREAD, USER, "remember: I use dark mode and prefer US English")
run_turn(THREAD, USER, "episode: billing dispute | issued pro-rata credit after tier mismatch")
out = run_turn(THREAD, USER, "What preferences should the UI use?")
print(out["messages"][-1].content[:500])

# Same user, new thread → long-term still loads
out2 = run_turn("demo-thread-2", USER, "Do you remember my language preference?")
print("--- new thread ---")
print(out2["messages"][-1].content[:500])

# Governance: list, edit, delete, clear (product surface)
print(run_turn(THREAD, USER, "/memory list")["messages"][-1].content[:1200])

# Hygiene demos (same thread): collapse transcript, then compact semantic atoms
run_turn(THREAD, USER, "/thread summarize")
run_turn(THREAD, USER, "remember: I also want metric units everywhere")
run_turn(THREAD, USER, "remember: No emojis in summaries")
run_turn(THREAD, USER, "/memory compact")
print(run_turn(THREAD, USER, "/memory list")["messages"][-1].content[:1200])


The UI should use:
- **Dark mode**
- **US English** locale/language conventions
--- new thread ---
Yes — you prefer **US English** (and you use **dark mode**).
**Your memories (stored for this user_id)**
- Semantic (1): 6359a05b…
    • [6359a05b-e3b2-4d8b-92f4-ee92454ec40a] I use dark mode and prefer US English
- Episodic (1): ebb5698f-487b-49ea-9fde-405b186518ac
    • billing dispute -> issued pro-rata credit after tier mismatch
- Procedural: Be concise and helpful. Prefer bullet points for comparisons.
**Your memories (stored for this user_id)**
- Semantic (1): consolid…
    • [consolidated_profile] - **Theme preference:** Use **dark mode**
- **Language preference:** **US English**
- **Units preference:** **Metric units everywhere**
- **Formatting preference:** **No emojis in summaries**
- Episodic (1): ebb5698f-487b-49ea-9fde-405b186518ac
    • billing dispute -> issued pro-rata credit after tier mismatch
- Procedural: Be concise and helpful. Prefer bullet points for comparisons.


## Persistence teaching graph (same primitives as production)

Checkpointers snapshot state at **super-step** boundaries. Below is the minimal two-node walkthrough from the docs (separate tiny graph so counts stay predictable).


In [ ]:
class PS(TypedDict):
    foo: str
    bar: Annotated[list[str], add]


def node_a(state: PS):
    return {"foo": "a", "bar": ["a"]}


def node_b(state: PS):
    return {"foo": "b", "bar": ["b"]}


ps_wf = StateGraph(PS)
ps_wf.add_node(node_a)
ps_wf.add_node(node_b)
ps_wf.add_edge(START, "node_a")
ps_wf.add_edge("node_a", "node_b")
ps_wf.add_edge("node_b", END)
persist_demo = ps_wf.compile(checkpointer=InMemorySaver())
ps_cfg: RunnableConfig = {"configurable": {"thread_id": "persistence-mini-demo"}}
persist_demo.invoke({"foo": "", "bar": []}, ps_cfg)
print("Checkpoint count:", len(list(persist_demo.get_state_history(ps_cfg))))


Checkpoint count: 4


## Serialization, encryption, memory poisoning

**CipherProtocol** + **EncryptedSerializer** let you swap in org-standard crypto. Optional cell below mirrors the persistence guide.


In [ ]:
from langgraph.checkpoint.serde.base import CipherProtocol
from langgraph.checkpoint.serde.encrypted import EncryptedSerializer

print("Extensibility: implement", CipherProtocol, "→ pass into", EncryptedSerializer)

key_hex = os.getenv("LANGGRAPH_AES_KEY", "")
if len(key_hex) == 64 and re.fullmatch(r"[0-9a-fA-F]+", key_hex):
    try:
        serde = EncryptedSerializer.from_pycryptodome_aes()
        enc_cp = InMemorySaver(serde=serde)
        g = StateGraph(PS)
        g.add_node(node_a)
        g.add_node(node_b)
        g.add_edge(START, "node_a")
        g.add_edge("node_a", "node_b")
        g.add_edge("node_b", END)
        enc_graph = g.compile(checkpointer=enc_cp)
        enc_graph.invoke({"foo": "", "bar": []}, {"configurable": {"thread_id": "encrypted-demo"}})
        print("Encrypted checkpointer round-trip: OK")
    except Exception as exc:
        print("Encrypted demo skipped:", exc)
else:
    print("Set LANGGRAPH_AES_KEY (64 hex chars) to exercise EncryptedSerializer.from_pycryptodome_aes().")


Extensibility: implement <class 'langgraph.checkpoint.serde.base.CipherProtocol'> → pass into <class 'langgraph.checkpoint.serde.encrypted.EncryptedSerializer'>
Set LANGGRAPH_AES_KEY (64 hex chars) to exercise EncryptedSerializer.from_pycryptodome_aes().


## Time travel on the **unified** app thread

Use `get_state_history` on the **same** `CHECKPOINTER` backing `agent_app` — this is how you debug production conversations or fork after HITL edits.


In [ ]:
cfg_tt: RunnableConfig = {"configurable": {"thread_id": THREAD}}
hist = list(agent_app.get_state_history(cfg_tt))
print("Unified thread snapshots:", len(hist))
if len(hist) >= 3:
    mid = hist[len(hist) // 2]
    print("Example checkpoint step:", mid.metadata.get("step"), "next:", mid.next)

# Mini-graph time travel (fork + replay) — same pattern on any compiled graph
class TT(TypedDict):
    foo: str
    bar: Annotated[list[str], add]


def tt_a(state: TT):
    return {"foo": "a", "bar": ["a"]}


def tt_b(state: TT):
    return {"foo": "b", "bar": ["b"]}


tt_wf = StateGraph(TT)
tt_wf.add_node("node_a", tt_a)
tt_wf.add_node("node_b", tt_b)
tt_wf.add_edge(START, "node_a")
tt_wf.add_edge("node_a", "node_b")
tt_wf.add_edge("node_b", END)
tt_graph = tt_wf.compile(checkpointer=InMemorySaver())
tt_cfg: RunnableConfig = {"configurable": {"thread_id": "time-travel-mini"}}
tt_graph.invoke({"foo": "", "bar": []}, tt_cfg)
before_b = next(s for s in tt_graph.get_state_history(tt_cfg) if s.next == ("node_b",))
forked = tt_graph.update_state(before_b.config, {"foo": "forked"}, as_node="node_a")
continued = tt_graph.invoke(None, forked)
print("Fork continue:", continued)
replay_cfg = {
    "configurable": {
        "thread_id": "time-travel-mini",
        "checkpoint_id": before_b.config["configurable"]["checkpoint_id"],
    }
}
print("Replay:", tt_graph.invoke(None, replay_cfg))


Unified thread snapshots: 32
Example checkpoint step: 14 next: ('__start__',)
Fork continue: {'foo': 'b', 'bar': ['a', 'b']}
Replay: {'foo': 'b', 'bar': ['a', 'b']}


## Hot path vs background writes

This notebook persists on the **hot path** (`node_persist`) for clarity. In production you can **enqueue** the same `UserMemoryService` calls from a worker to reduce latency, then tune **debouncing** and **compaction** (`/memory compact`, scheduled jobs) so quality stays high.


## Production checklist

| Swap | Where |
| --- | --- |
| Durability | `build_checkpointer()` → SQLite / Postgres savers |
| Multi-process | Shared DB for checkpoints + store ([Postgres store](https://docs.langchain.com/oss/python/langgraph/persistence#memory-store)) |
| User memory UI | HTTP handlers call `UserMemoryService` (list/edit/delete/clear/summarize) |
| Safety | AuthZ on `user_id`, sanitize store payloads, encrypt at rest, monitor for poisoning |

Treat store and checkpoint payloads as **untrusted text** at the application boundary.
